In [5]:
from pathlib import Path

dataset_dir = Path(r"E:\kaggle_cache\datasets\hari31416\food-101\versions\1\food-101")
print("Local Kaggle dataset path:", dataset_dir)
print("Dataset exists:", dataset_dir.is_dir())

Local Kaggle dataset path: E:\kaggle_cache\datasets\hari31416\food-101\versions\1\food-101
Dataset exists: True


In [6]:
from pathlib import Path
import copy
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2

print("PyTorch:", torch.__version__)
print("torchvision:", __import__("torchvision").__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

PyTorch: 2.15.0+cu134
torchvision: 0.29.2+cu134
CUDA available: True
Using: cuda


In [7]:
dataset_dir = Path(r"E:\kaggle_cache\datasets\hari31416\food-101\versions\1\food-101")
train_dir = dataset_dir / "train"
validation_dir = dataset_dir / "validation"

print("Dataset:", dataset_dir)
print("Train exists:", train_dir.is_dir())
print("Validation exists:", validation_dir.is_dir())

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder(str(train_dir), transform=train_transform)
validation_data = datasets.ImageFolder(str(validation_dir), transform=eval_transform)

if train_data.class_to_idx != validation_data.class_to_idx:
    raise RuntimeError("Train and validation class mappings differ")

class_names = train_data.classes
num_classes = len(class_names)
batch_size = 128 if torch.cuda.is_available() else 16

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
    pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    validation_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    persistent_workers=True,
    pin_memory=torch.cuda.is_available(),
)

print("Classes:", num_classes)
print("Training images:", len(train_data))
print("Validation images:", len(validation_data))

Dataset: E:\kaggle_cache\datasets\hari31416\food-101\versions\1\food-101
Train exists: True
Validation exists: True
Classes: 101
Training images: 95950
Validation images: 5050


In [8]:
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 3
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print("Model created; starting training is the next step")

Model created; starting training is the next step


In [9]:
history = []
best_model_state = copy.deepcopy(model.state_dict())
best_validation_accuracy = 0.0
train_batches = len(train_loader)
validation_batches = len(validation_loader)

print(f"Starting training: {epochs} epochs, {train_batches} train batches/epoch, {validation_batches} validation batches/epoch", flush=True)

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    print(f"Epoch {epoch}/{epochs}: training started", flush=True)

    for batch_number, (images, labels) in enumerate(train_loader, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * labels.size(0)
        running_correct += (logits.argmax(dim=1) == labels).sum().item()

        if batch_number == 1 or batch_number % 100 == 0:
            print(f"  Batch {batch_number}/{train_batches}", flush=True)

    print(f"Epoch {epoch}/{epochs}: validation started", flush=True)
    model.eval()
    validation_correct = 0
    with torch.no_grad():
        for batch_number, (images, labels) in enumerate(validation_loader, start=1):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            predictions = model(images).argmax(dim=1)
            validation_correct += (predictions == labels).sum().item()
            if batch_number == 1 or batch_number % 100 == 0:
                print(f"  Validation batch {batch_number}/{validation_batches}", flush=True)

    train_loss = running_loss / len(train_data)
    train_accuracy = running_correct / len(train_data)
    validation_accuracy = validation_correct / len(validation_data)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_accuracy": validation_accuracy,
    })

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_model_state = copy.deepcopy(model.state_dict())

    scheduler.step()
    print(
        f"Epoch {epoch}/{epochs}: "
        f"loss={train_loss:.4f}, train_acc={train_accuracy:.4f}, "
        f"validation_acc={validation_accuracy:.4f}"
    )

Starting training: 3 epochs, 750 train batches/epoch, 40 validation batches/epoch
Epoch 1/3: training started
  Batch 1/750
  Batch 100/750
  Batch 200/750
  Batch 300/750
  Batch 400/750
  Batch 500/750
  Batch 600/750
  Batch 700/750
Epoch 1/3: validation started
  Validation batch 1/40
Epoch 1/3: loss=nan, train_acc=0.0099, validation_acc=0.0099
Epoch 2/3: training started
  Batch 1/750
  Batch 100/750
  Batch 200/750
  Batch 300/750
  Batch 400/750
  Batch 500/750
  Batch 600/750
  Batch 700/750
Epoch 2/3: validation started


RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "e:\cvllm\projects\.venv\Lib\site-packages\torch\utils\data\_utils\worker.py", line 375, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "e:\cvllm\projects\.venv\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ~~~~~~~~~~~~~~~^^^^^^
  File "e:\cvllm\projects\.venv\Lib\site-packages\torch\utils\data\_utils\collate.py", line 401, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
  File "e:\cvllm\projects\.venv\Lib\site-packages\torch\utils\data\_utils\collate.py", line 215, in collate
    collate(samples, collate_fn_map=collate_fn_map)
    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\cvllm\projects\.venv\Lib\site-packages\torch\utils\data\_utils\collate.py", line 155, in collate
    return collate_fn_map[elem_type](bat

In [ ]:
checkpoint_path = Path("mobilenet_food101_best.pth")
metrics_path = Path("mobilenet_food101_metrics.json")

torch.save({
    "architecture": "mobilenet_v2",
    "num_classes": num_classes,
    "class_names": class_names,
    "class_to_idx": train_data.class_to_idx,
    "model_state_dict": best_model_state,
    "validation_accuracy": best_validation_accuracy,
    "history": history,
}, checkpoint_path)
metrics_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

print("Saved checkpoint:", checkpoint_path.resolve())
print("Saved metrics:", metrics_path.resolve())
print("Best validation accuracy:", best_validation_accuracy)